# Thêm Thư Viện

In [165]:
import pyodbc
import pandas as pd

# Tạo kết nối

In [166]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_lib = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=dwh_lib;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)


# ETL bảng Dim_Date

## Xóa data bảng DIM_date 

In [51]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Date"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ file CSV

In [ ]:
df_Data_Dim_Date = pd.read_csv("./Source-Data/Dim_Date.csv")
print(df_Data_Dim_Date)

       Date_key   Full_date                     Date_text   Day_name  \
0      19700101    1/1/1970     Thursday, January 1, 1970   Thursday   
1      19700102    1/2/1970       Friday, January 2, 1970     Friday   
2      19700103    1/3/1970     Saturday, January 3, 1970   Saturday   
3      19700104    1/4/1970       Sunday, January 4, 1970     Sunday   
4      19700105    1/5/1970       Monday, January 5, 1970     Monday   
...         ...         ...                           ...        ...   
29215  20491227  12/27/2049     Monday, December 27, 2049     Monday   
29216  20491228  12/28/2049    Tuesday, December 28, 2049    Tuesday   
29217  20491229  12/29/2049  Wednesday, December 29, 2049  Wednesday   
29218  20491230  12/30/2049   Thursday, December 30, 2049   Thursday   
29219  20491231  12/31/2049     Friday, December 31, 2049     Friday   

       Week_of_quarter  Day_of_week  Month  Quarter  Year  Day  
0                    1            4      1        1  1970    1  
1    

## Load data vào dwh_lib

In [ ]:

for index, row in df_Data_Dim_Date.iterrows():
    cursor = conn_dwh_lib.cursor()
    insert_query = """
    INSERT INTO DIM_Date (Date_key, Full_date, Date_text, Day, Week_of_quarter, Month, Quarter, Year, Day_of_week, Day_name)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    cursor.execute(insert_query, row['Date_key'], row['Full_date'], row['Date_text'], row['Day'], 
                   row['Week_of_quarter'], row['Month'], row['Quarter'], row['Year'], row['Day_of_week'], row['Day_name'])
    conn_dwh_lib.commit()
    cursor.close()

# ETL bảng Dim_Khoa

## Xóa data bảng DIM_Khoa

In [217]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Khoa"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ CSV

In [219]:
df_Data_Khoa = pd.read_csv("./Source-Data/Dim_Khoa.csv")
print(df_Data_Khoa)

    Mã                             Tên khoa
0    1               Khoa Lý luận Chính trị
1    2               Khoa Khoa học ứng dụng
2    3              Khoa Cơ khí Chế tạo máy
3    4                  Khoa Điện - Điện tử
4    5                 Khoa Cơ khí Động Lực
5    6                         Khoa Kinh tế
6    7             Khoa Công nghệ thông tin
7    8              Khoa In và Truyền thông
8    9     Khoa Công nghệ May và Thời Trang
9   10  Khoa Công nghệ Hóa học và Thực phẩm
10  11                        Khoa Xây dựng
11  12                       Khoa Ngoại ngữ
12  13          Khoa Đào tạo Chất lượng cao
13  14                Viện Sư phạm Kỹ thuật
14  15  Trường Trung học Kỹ thuật Thực hành


## Load data vào bảng Dim_Khoa

In [220]:
cursor_dwh = conn_dwh_lib.cursor()

insert_query = """
INSERT INTO DIM_Khoa (ID_khoa, Ten_khoa) VALUES (?, ?)
"""

for index, row in df_Data_Khoa.iterrows():
    values = (row['Mã'], row['Tên khoa'])
    cursor_dwh.execute(insert_query, values)
    
conn_dwh_lib.commit()

# ETL bảng Dim_Dan_Toc

## Xóa data bảng DIM_Dan_Toc

In [167]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Dan_toc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data

### Đọc data từ CSV

In [168]:
df_Data_Dim_Dan_toc = pd.read_csv("./Source-Data/Dim_Dan_toc.csv")
df_Data_Dim_Dan_toc = df_Data_Dim_Dan_toc.where(pd.notnull(df_Data_Dim_Dan_toc), None)
print(df_Data_Dim_Dan_toc)

    Mã               Tên                                       Tên gọi khác
0    1              Kinh                                               Việt
1    2               Tày          Thổ, Ngạn, Phén, Thù Lao, Pa Dí, Tày Khao
2    3              Thái  Tày Đăm, Tày Mười, Tày Thanh, Mán Thanh, Hàng ...
3    4               Hoa  Hán, Triều Châu, Phúc Kiến, Quảng Đông, Hải Na...
4    5            Khơ-me             Cur, Cul, Cu, Thổ, Việt gốc Miên, Krôm
5    6             Mường               Mol, Mual, Mọi, Mọi Bi, Ao Tá, Ậu Tá
6    7              Nùng  Xuồng, Giang, Nùng An, Phàn Sinh, Nùng Cháo, N...
7    8             HMông  Mèo, Hoa, Mèo Xanh, Mèo Đỏ, Mèo Đen, Ná Mẻo, M...
8    9               Dao  Mán, Động, Trại, Xá, Dìu, Miên, Kiềm, Miền, Qu...
9   10           Gia-rai    Giơ-rai, Tơ-buăn, Chơ-rai, Hơ-bau, Hđrung, Chor
10  11              Ngái                            Xín, Lê, Đản, Khách Gia
11  12              Ê-đê  Ra-đê, Đê, Kpạ, A-đham, Krung, Ktul, Đliê Ruê,...
12  13      

### Đọc data từ SQL Server

In [203]:
query_dan_toc = "SELECT Id, dbo.DecodeUTF8String(Dan_toc) AS Dan_toc FROM Dan_toc "
df_dan_toc = pd.read_sql(query_dan_toc, conn_libol)
print(df_dan_toc)

    Id  Dan_toc
0    1     Kinh
1    2    Mường
2    3      Tày
3    4     Thái
4    5      Hoa
..  ..      ...
68  71     Ê Đê
69  72      Thổ
70  73    Kờ Ho
71  74     Jrai
72  75  Châu mạ

[73 rows x 2 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_17252\1986067178.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dan_toc = pd.read_sql(query_dan_toc, conn_libol)


## Xử lý data

In [205]:
df_dan_toc['Mapping Mã'] = None
df_dan_toc['CSV Mã'] = df_Data_Dim_Dan_toc['Mã']
df_dan_toc['CSV Tên'] = df_Data_Dim_Dan_toc['Tên']
df_dan_toc['CSV Tên khác'] = df_Data_Dim_Dan_toc['Tên gọi khác'].str.lower()
for i, dan_toc in enumerate(df_dan_toc['Dan_toc']):
    dan_toc = dan_toc.lower()
    dan_toc_bogach = dan_toc.replace("-", " ")
    dan_toc_botrong = dan_toc.replace(" ", "-")

    for j, row in df_dan_toc.iterrows():
        CSV_ten = row['CSV Tên'].lower() if pd.notna(row['CSV Tên']) else ""
        CSV_ten_khac = row['CSV Tên khác'].lower() if pd.notna(row['CSV Tên khác']) else ""
        if ((pd.notna(CSV_ten) and dan_toc == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc in CSV_ten_khac) or 
            (pd.notna(CSV_ten) and dan_toc_bogach == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_bogach in CSV_ten_khac) or
            (pd.notna(CSV_ten) and dan_toc_botrong == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_botrong in CSV_ten_khac)):
            df_dan_toc.at[i, 'Mapping Mã'] = row['CSV Mã']
            break
        else:
            df_dan_toc.at[i, 'Mapping Mã'] = 56
print(df_dan_toc)
df_dan_toc.to_csv('./Source-Data/Mapping_Dan_toc.csv', index=False, encoding='utf-8-sig')


    Id  Dan_toc Mapping Mã  CSV Mã CSV Tên  \
0    1     Kinh        1.0     1.0    Kinh   
1    2    Mường        3.0     2.0     Tày   
2    3      Tày        2.0     3.0    Thái   
3    4     Thái        3.0     4.0     Hoa   
4    5      Hoa        4.0     5.0  Khơ-me   
..  ..      ...        ...     ...     ...   
68  71     Ê Đê       12.0     NaN     NaN   
69  72      Thổ        2.0     NaN     NaN   
70  73    Kờ Ho         56     NaN     NaN   
71  74     Jrai         56     NaN     NaN   
72  75  Châu mạ       28.0     NaN     NaN   

                                         CSV Tên khác  
0                                                việt  
1           thổ, ngạn, phén, thù lao, pa dí, tày khao  
2   tày đăm, tày mười, tày thanh, mán thanh, hàng ...  
3   hán, triều châu, phúc kiến, quảng đông, hải na...  
4              cur, cul, cu, thổ, việt gốc miên, krôm  
..                                                ...  
68                                                NaN  

## Load data

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()

insert_query = """
INSERT INTO DIM_Dan_toc (ID_dan_toc, Dan_toc, Ten_khac) VALUES (?, ?, ?)
"""

for index, row in df_Data_Dim_Dan_toc.iterrows():
    values = (row['Mã'], row['Tên'], row['Tên gọi khác'])
    cursor_dwh.execute(insert_query, values)
    
conn_dwh_lib.commit()

# ETL bảng Dim_Trinh_do

## Xóa data bảng Dim_Trinh_do

In [211]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Trinh_do"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [214]:
query_Trinh_do = "SELECT trinh_do_id, dbo.DecodeUTF8String(loai_trinh_do) AS Trinh_do FROM Trinh_do "
df_Trinh_do = pd.read_sql(query_Trinh_do, conn_libol)
print(df_Trinh_do)

   trinh_do_id                 Trinh_do
0           10                   PGS.TS
1            3                 Cao đẳng
2            4                  Đại học
3            5                  Thạc sĩ
4            6                  Tiến sĩ
5            7              Phó tiến sĩ
6           11  Trung học chuyên nghiệp
7            9             Trung học PT
8           12                    12/12


C:\Users\admin\AppData\Local\Temp\ipykernel_17252\57140960.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Trinh_do = pd.read_sql(query_Trinh_do, conn_libol)


## Load data vào bảng Dim

In [216]:
cursor_dwh = conn_dwh_lib.cursor()

insert_query = """
INSERT INTO DIM_Trinh_do (ID_trinh_do, Loai_trinh_do) VALUES (?, ?)
"""

for index, row in df_Trinh_do.iterrows():
    values = (row['trinh_do_id'], row['Trinh_do'])
    cursor_dwh.execute(insert_query, values)
    
conn_dwh_lib.commit()

# ETL bảng 